In [633]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

base_url = "http://127.0.0.1:8045"
api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
client = Anthropic(api_key=api_key, base_url=base_url)
# model = "gemini-3-flash"
model = "claude-opus-4-5-thinking"

In [634]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [636]:
messages = []

add_user_message(messages, "3x+5=14?,求x")

chat(messages, thinking=True)

Message(id='req_vrtx_011CY9MQPG2vqZ5Eip8HbXRJ', content=[ThinkingBlock(signature=None, thinking='Simple algebra problem.\n\n3x + 5 = 14\n3x = 9\nx = 3', type='thinking'), TextBlock(citations=None, text='## 求解方程 3x + 5 = 14\n\n**步骤：**\n\n1. **两边减去 5：**\n   $$3x + 5 - 5 = 14 - 5$$\n   $$3x = 9$$\n\n2. **两边除以 3：**\n   $$x = \\frac{9}{3}$$\n\n$$\\boxed{x = 3}$$\n\n**验证：** 3 × **3** + 5 = 9 + 5 = 14 ✅', type='text')], model='claude-opus-4-6-thinking', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=149, output_tokens=174, server_tool_use=None, service_tier=None))

请求报文：
```json
{
  "max_tokens": 4000,
  "messages": [
    {
      "role": "user",
      "content": "3x+5=14?,求x"
    }
  ],
  "model": "claude-opus-4-5-thinking",
  "stop_sequences": [],
  "temperature": 1,
  "thinking": {
    "type": "enabled",
    "budget_tokens": 1024
  }
}
```

响应报文
```json
{
  "id": "req_vrtx_011CY9MQPG2vqZ5Eip8HbXRJ",
  "type": "message",
  "role": "assistant",
  "model": "claude-opus-4-6-thinking",
  "content": [
    {
      "type": "thinking",
      "thinking": "Simple algebra problem.\n\n3x + 5 = 14\n3x = 9\nx = 3"
    },
    {
      "type": "text",
      "text": "## 求解方程 3x + 5 = 14\n\n**步骤：**\n\n1. **两边减去 5：**\n   $$3x + 5 - 5 = 14 - 5$$\n   $$3x = 9$$\n\n2. **两边除以 3：**\n   $$x = \\frac{9}{3}$$\n\n$$\\boxed{x = 3}$$\n\n**验证：** 3 × **3** + 5 = 9 + 5 = 14 ✅"
    }
  ],
  "stop_reason": "end_turn",
  "usage": {
    "input_tokens": 149,
    "output_tokens": 174,
    "cache_read_input_tokens": 0,
    "cache_creation_input_tokens": 0
  }
}
```